In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


In [2]:
df = pd.read_csv('../data/sample.csv', delimiter=';', encoding='utf-8')
df_train = df[df['ds_notpriced'].notna()]
df_train.head(10)

,ds_text,ds_reason,ds_notpriced,macro_ds_notpriced
0,app_filter:Qsa nok penumper 7555666,Docto nok,QSA nok,Docto nok
1,app_filter:Api nok penumper 1234567,Docto nok,API nok,Docto nok
2,spread not registered penumper 1593578,spread not registered,spread not registered,spread not registered
3,counterparty not registered penumper 1559357,client not registered,client not registered,client not registered
5,app_filter:Qsa nok penumper 9636852,Docto nok,QSA nok,Docto nok
6,spread not registered penumper 5644423,spread not registered,spread not registered,spread not registered
8,app_filter:Suitability nok penumper 8889996,Docto nok,Suitability nok,Docto nok


In [3]:
import re
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def limpar(texto):
    t = str(texto).lower()
    t = re.sub(r'\b\d+\b', 'idnum', t)
    t = re.sub(r'[_:()\[\]]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df_train = df_train.copy()
df_train['ds_text_clean'] = df_train['ds_text'].apply(limpar)

# padroniza rótulos (pega o "registerered", espaços extras etc.)
for col in ['ds_notpriced', 'macro_ds_notpriced']:
    df_train[col] = (df_train[col].str.strip()
                     .str.replace('registerered', 'registered'))

# confere se cada rótulo detalhado aponta pra um único macro
checagem = df_train.groupby('ds_notpriced')['macro_ds_notpriced'].nunique()
print(checagem[checagem > 1])   # se vier vazio, o mapeamento é consistente

mapa_macro = (df_train.drop_duplicates('ds_notpriced')
              .set_index('ds_notpriced')['macro_ds_notpriced']
              .to_dict())

Series([], Name: macro_ds_notpriced, dtype: int64)


In [4]:
modelo = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    LogisticRegression(max_iter=1000, class_weight='balanced')
)
modelo.fit(df_train['ds_text_clean'], df_train['ds_notpriced'])

,steps,"[('tfidfvectorizer', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [5]:
LIMIAR

NameError: name 'LIMIAR' is not defined

In [ ]:
df_novo = df[df['ds_notpriced'].isna()]
df_novo = df_novo.copy()   # sua amostra sem rótulo
df_novo['ds_text_clean'] = df_novo['ds_text'].apply(limpar)

proba = modelo.predict_proba(df_novo['ds_text_clean'])

df_novo['ds_notpriced_pred'] = modelo.predict(df_novo['ds_text_clean'])
df_novo['macro_ds_notpriced_pred'] = df_novo['ds_notpriced_pred'].map(mapa_macro)
df_novo['confianca'] = proba.max(axis=1)

# casos em que o modelo está inseguro -> revisão manual
LIMIAR = 0.7
df_novo['revisar'] = df_novo['confianca'] < LIMIAR

df_novo[['ds_text', 'ds_notpriced_pred', 'macro_ds_notpriced_pred',
         'confianca', 'revisar']].head(20)

,ds_text,ds_notpriced_pred,macro_ds_notpriced_pred,confianca,revisar
4,app_filter:Suitability nok penumper 8799453,Suitability nok,Docto nok,0.348536,True
7,spread not registered penumper 7451965,spread not registered,spread not registered,0.357791,True
